In [22]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.tools import Tool
from langchain_classic.agents import create_engine, AgentType
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit
from sqlalchemy import create_engine
import sqlite3

In [23]:
import sqlite3

# Connect to database (will be created if not exists)
conn = sqlite3.connect("../data/company.db")

cursor = conn.cursor()

# Create table
cursor.execute("""
CREATE TABLE IF NOT EXISTS employees(
    id INTEGER PRIMARY KEY,
    name TEXT,
    department TEXT,
    salary INTEGER
)
""")

# Insert sample data
cursor.executemany("""
INSERT INTO employees(name, department, salary)
VALUES (?, ?, ?)
""", [
    ("John", "AI", 90000),
    ("Sara", "ML", 95000),
    ("Mike", "Data", 80000),
    ("Anna", "AI", 105000)
])

conn.commit()
conn.close()

# Connect Database with Langchain 

In [24]:
from sqlalchemy import create_engine

engine = create_engine("sqlite:///../data/company.db")
db = SQLDatabase(engine)

part 2

In [25]:
loader = PyMuPDFLoader("/Users/yashshakya787/genAI/data/customer_support_kb.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50,
    separators    = ["\n\n", "\n", "Q:", "A:", " "]
)
chunks = splitter.split_documents(docs)
print(f" Total chunks: {len(chunks)}")

 Total chunks: 8


In [26]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text"
)

create the vector database

In [27]:
vectorstores = FAISS.from_documents(
    chunks,
    embeddings
)
retrivier = vectorstores.as_retriever()

# Part 3 create tools

# tool 1--- RAG Retriever

In [28]:
def rag_tool(query):
    docs = retrivier.invoke(query)

    return docs[0].page_content

In [29]:
rag = Tool(
    name= "RAG_Document_Search",
    func=rag_tool,
    description="Searches Knowledge base documents"
)

# Tool 2 --- SQL Database Tool

In [30]:
toolkit = SQLDatabaseToolkit(db=db)
sql_tools = toolkit.get_tools()

ValidationError: 1 validation error for SQLDatabaseToolkit
llm
  Field required [type=missing, input_value={'db': <langchain_communi... object at 0x149a281f0>}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.12/v/missing

# 3 calculator tool

In [31]:
def calculator(query):
    try:
        return str(eval(query))
    except:
            return "Invalid math expression"

In [ ]:
calculator_tool = Tool(
    name = "calculator",
    func = calculator,
    description="usefull for math calculations"
)